# Лекция: Работа с данными и коэффициент корреляции

**Дисциплина:** Введение в анализ больших данных

В этой лекции:
- генерация выборок и базовые статистики;
- эмпирическая функция распределения (ECDF);
- корреляция Пирсона, Спирмена, Кендалла и проверка значимости;
- стандартизация и логарифмирование;
- точечные диаграммы и группировка на реальном датасете.

Примеры **не совпадают** с лабораторным заданием (другие параметры генерации и другой датасет). Цель — освоить методы; задание выполните самостоятельно.


## 0. Импорт библиотек


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.distributions.empirical_distribution import ECDF

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")
np.random.seed(7)

print("Библиотеки загружены")


---
## 1. Нормальная выборка

Генерация: `stats.norm.rvs(loc=μ, scale=σ, size=n)`.

**Демо:** «рост» $N(170,\,8)$, $n = 80$.


In [ ]:
height = stats.norm.rvs(loc=170, scale=8, size=80)

print("Первые 8:", height[:8].round(2))
print(f"mean = {height.mean():.3f}")
print(f"sd   = {height.std(ddof=1):.3f}")
print(f"var  = {height.var(ddof=1):.3f}")


---
## 2. Дискретная выборка (биномиальная)

`stats.binom.rvs(n=trials, p=prob, size=m)`.

**Демо:** число успехов в 12 испытаниях, $p=0.4$, объём 200.


In [ ]:
successes = stats.binom.rvs(n=12, p=0.4, size=200)

print("Первые 15:", successes[:15])
print(pd.Series(successes).describe())


---
## 3. Эмпирическая функция распределения (ECDF)

ECDF — доля наблюдений ≤ x.  
Строится через `ECDF` (statsmodels) или `sns.ecdfplot`.


In [ ]:
ecdf_h = ECDF(height)
ecdf_s = ECDF(successes)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].step(ecdf_h.x, ecdf_h.y, where="post", color="steelblue", label="height")
axes[0].step(ecdf_s.x, ecdf_s.y, where="post", color="darkorange", label="successes")
axes[0].set_title("ECDF")
axes[0].set_xlabel("значение")
axes[0].set_ylabel("F(x)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(height, bins=15, density=True, alpha=0.6, color="steelblue", label="height")
axes[1].hist(successes, bins=range(0, 14), density=True, alpha=0.6,
             color="darkorange", align="left", label="successes")
axes[1].set_title("Гистограммы")
axes[1].set_xlabel("значение")
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 4. Корреляция: Пирсон, Спирмен, Кендалл

| Метод | Когда уместен |
|-------|----------------|
| **Пирсон** | линейная связь, количественные ≈ нормальные |
| **Спирмен** | монотонная связь, ранги / порядковые |
| **Кендалл** | порядковые шкалы, мало совпадающих рангов |

|r|: 0–0.1 нет связи; 0.1–0.5 слабая; 0.5–0.7 умеренная; 0.7–1 сильная.

H0 в тесте: «связи нет» (ρ = 0). При p < 0.05 обычно отвергают H0.


In [ ]:
np.random.seed(11)
x = stats.norm.rvs(0, 1, size=120)
y = 0.6 * x + stats.norm.rvs(0, 0.8, size=120)

r_p, p_p = stats.pearsonr(x, y)
r_s, p_s = stats.spearmanr(x, y)
r_k, p_k = stats.kendalltau(x, y)

print(f"Пирсон  : r = {r_p:.4f}, p = {p_p:.4g}")
print(f"Спирмен : r = {r_s:.4f}, p = {p_s:.4g}")
print(f"Кендалл : r = {r_k:.4f}, p = {p_k:.4g}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.7, edgecolors="black", s=40)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Scatter: Пирсон r = {r_p:.3f}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 5. Матрица и корреляции между столбцами

`np.column_stack` / `reshape` + `np.corrcoef` или `DataFrame.corr()`.


In [ ]:
M2 = np.column_stack([x, y])
print("Форма M2:", M2.shape)

long = stats.norm.rvs(size=200)
M4 = long.reshape(50, 4)
corr4 = np.corrcoef(M4, rowvar=False)
print("Корреляции 4 столбцов:\n", np.round(corr4, 3))

plt.figure(figsize=(5, 4))
sns.heatmap(corr4, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0,
            xticklabels=[f"c{i+1}" for i in range(4)],
            yticklabels=[f"c{i+1}" for i in range(4)])
plt.title("Матрица корреляций")
plt.tight_layout()
plt.show()


---
## 6. Стандартизация

$$
z = \frac{x - \bar{x}}{s}
$$

После преобразования mean ≈ 0, sd ≈ 1.


In [ ]:
z_manual = (successes - successes.mean()) / successes.std(ddof=1)
z_sklearn = (successes - np.mean(successes)) / np.std(successes, ddof=1)

print("Первые 5 (вручную):", z_manual[:5].round(4))
print("Совпадают?", np.allclose(z_manual, z_sklearn))
print(f"mean(z) ≈ {z_manual.mean():.2e}, sd(z) ≈ {z_manual.std(ddof=1):.4f}")


---
## 7. Логарифмирование

Полезно при правосторонней асимметрии. Берите только положительные значения.


In [ ]:
positive = height[height > 0]
log_h = np.round(np.log(positive), 3)
print("log(height), первые 8:", log_h[:8])


---
## 8. Пример на датасете **penguins**

Возьмём датасет пингвинов: масса тела, вид, остров.


In [ ]:
penguins = sns.load_dataset("penguins").dropna()
print(penguins.head())
print(penguins.columns.tolist())


### Dotchart: масса тела по особям (с группировкой по виду)


In [ ]:
df = penguins.sort_values("body_mass_g")
palette = {"Adelie": "C0", "Chinstrap": "C1", "Gentoo": "C2"}
colors = df["species"].map(palette)

plt.figure(figsize=(9, 10))
plt.scatter(df["body_mass_g"], range(len(df)), c=colors, s=35, edgecolors="none")
plt.yticks([])
plt.xlabel("body_mass_g")
plt.title("Масса тела пингвинов (dotchart-стиль)\nцвет = вид")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=s)
           for s, c in palette.items()]
plt.legend(handles=handles, title="species")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


### Описательные статистики по группам


In [ ]:
print("=== body_mass_g (вся выборка) ===")
print(penguins["body_mass_g"].describe())

print("\n=== по species ===")
print(penguins.groupby("species")["body_mass_g"].describe().round(1))


### Гистограммы и ECDF (вся выборка и по группам)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].hist(penguins["body_mass_g"], bins=20, color="steelblue",
                edgecolor="black", alpha=0.7)
axes[0, 0].set_title("Гистограмма body_mass_g")

ec = ECDF(penguins["body_mass_g"])
axes[0, 1].step(ec.x, ec.y, where="post", color="steelblue")
axes[0, 1].set_title("ECDF body_mass_g")
axes[0, 1].set_ylabel("F(x)")

for sp, col in palette.items():
    sub = penguins.loc[penguins["species"] == sp, "body_mass_g"]
    axes[1, 0].hist(sub, bins=12, alpha=0.5, label=sp, color=col, edgecolor="black")
axes[1, 0].set_title("Гистограмма по species")
axes[1, 0].legend()

for sp, col in palette.items():
    sub = penguins.loc[penguins["species"] == sp, "body_mass_g"]
    ec_s = ECDF(sub)
    axes[1, 1].step(ec_s.x, ec_s.y, where="post", label=sp, color=col)
axes[1, 1].set_title("ECDF по species")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


In [ ]:
penguins = penguins.copy()
penguins["log_mass"] = np.log(penguins["body_mass_g"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(penguins["log_mass"], bins=20, color="purple", edgecolor="black", alpha=0.7)
axes[0].set_title("Гистограмма log(body_mass_g)")
ec_log = ECDF(penguins["log_mass"])
axes[1].step(ec_log.x, ec_log.y, where="post", color="purple")
axes[1].set_title("ECDF log(body_mass_g)")
plt.tight_layout()
plt.show()


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Нормальная выборка | `stats.norm.rvs(loc, scale, size)` |
| Биномиальная | `stats.binom.rvs(n=trials, p=prob, size=m)` |
| mean / sd / var | `.mean()`, `.std(ddof=1)`, `.var(ddof=1)` |
| summary | `pd.Series(x).describe()` |
| ECDF | `ECDF(x)` (statsmodels) / `sns.ecdfplot` |
| Пирсон | `stats.pearsonr(x, y)` → (r, p) |
| Спирмен | `stats.spearmanr(x, y)` |
| Кендалл | `stats.kendalltau(x, y)` |
| Стандартизация | `(x - x.mean()) / x.std(ddof=1)` |
| Логарифм + округление | `np.round(np.log(x), 3)` |
| Корреляционная матрица | `np.corrcoef(...)` / `df.corr()` |
| Dotchart-стиль | `plt.scatter` + сортировка + цвет по группе |

---
## Что сделать после лекции

1. Повторите генерацию и корреляции с **другими** параметрами.
2. Откройте лабораторное задание и выполните его **самостоятельно** (свои n, mean, sd; свой датасет из задания).
3. Смотрите и на коэффициент, и на p-value; scatter помогает увидеть форму связи.

Удачи!
